## Imports and setup

This notebook is in a customized project and requires a custumized python module (.py file) to run. By default the path to the start module for a custom project is ./src/"project_name"; it needs to contain a startup function. The default name of that function is "Run_process". In this project, [AI4SH](https://ai4soilhealth.eu), the path to the custom project is ./src/ai4sh and the first cell calls the Run_process function in that package.

This notebook is the single-step counterpart to `load_ai4sh_wetlab_data.ipynb`, covering the Agrolab wetlab pipeline. It replaces the original's 1 translate cell + 2 manage cells (observation_log, observation) with 2 insert cells (translate and insert combined per step), in the same order and split the same way as the manage cells in `load_ai4sh_wetlab_data.ipynb` - this keeps errors traceable to a single step instead of one combined observation_log + observation run. This route is INSERT-only - use `load_ai4sh_wetlab_data.ipynb` instead if you need to UPDATE existing records.

In [ ]:
# Standard library imports
from os import path, getcwd

import sys

# Get the working directory of the notebook
notebook_path = getcwd()

# Set the path to the root directory of this jupyter notebook (python) project
module_path = path.abspath(path.join('../..'))

if module_path not in sys.path:

    sys.path.append(module_path)

# Package application imports
from src.lib import Initiate_process

# Import the custom class for running processes defined in the ai4sh package
from src.ai4sh import Run_process

## Scheme file
To run this cell you need to set the [scheme_file](https://xspatula.github.io/setup_core_db_docs/framework/scheme_file/), that defines the location (path) to the project and associated process files and your user. You can set the path to the scheme file with a relative path (relative to this notebook), in your home directory using tilde (~) or with an absolute path in the syntax of your operating system. 

In [ ]:
# Set the path to the scheme_file

scheme_file = '../scheme_ai4sh.json'

## Insert eDNA observation log

Translates and inserts the eDNA observation log.

**process**: [insert_tabular_data](add_link_to_documentation)

**process_file**: [insert_eDNA_observation_log.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/eDNA/insert_process/insert_eDNA_observation_log.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    Run_process(structured_process_D, scheme_params_D)

## Insert eDNA observation

Translates and inserts the Agrolab observation values.

**process**: [insert_tabular_data](add_link_to_documentation)

**process_file**: [insert_eDNA_observation.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/eDNA/insert_process/insert_eDNA_observation.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    Run_process(structured_process_D, scheme_params_D)

## Insert eDNA ASV abundances (Stage 4b)

Bulk loads the ASV files written by `prepare_ai4sh_data/edna_final_results_to_xspatula.py` into `observation.edna_asv` (one row per ASV and method pipeline, linked to its deepest resolved taxon) and `observation.edna_asv_abundance` (relative abundance and rarefied read count per observation), using postgres `COPY`.

Requires: the observations above, and the taxa (Stage 4a, `insert_ai4sh_taxon_data.ipynb`). Nothing is loaded if an observation or a taxon is missing, or if an ASV key already in the database now carries another lineage (the laboratory changed the row order). Insert only; rerunning inserts nothing.

**process**: insert_tabular_data (target process manage_edna_asv)

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/eDNA/insert_process/insert_asv_prokaryotes.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    Run_process(structured_process_D, scheme_params_D)

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/eDNA/insert_process/insert_asv_fungi.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    Run_process(structured_process_D, scheme_params_D)